# Title


**IMDB Movie Review Sentiment Analysis**

# Problem Statement



Nowadays, people share thousands of movie reviews online. These reviews contain different opinions, such as positive and negative feedback.

Manually reading and identifying whether a review is positive or negative takes a lot of time.

Therefore, the problem is to develop a Machine Learning model that automatically analyzes movie reviews and classifies them as Positive or Negative based on the text.

# Main Objective

1. Collect movie reviews from the IMDB dataset.

2. Clean and preprocess the review text.

3. Convert text into numerical features using Word2Vector.

4. Train different Machine Learning classification algorithms.

5. Compare their performance using F1 Score.

6. Select the best-performing model using hyperparameter tuning.

7. Predict whether a new movie review is Positive or Negative.

# Dataset Loading and Data Overivew

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
!pip install optuna gensim

In [40]:
import pandas as pd
import numpy as np
import re
import html
import nltk
import pandas as pd

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.pipeline import Pipeline
from gensim.models import Word2Vec
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,BaggingClassifier,AdaBoostClassifier, GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import f1_score, classification_report

# Download required resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [20]:
df = pd.read_csv('/content/drive/MyDrive/ML/imdb sentiment analysis/IMDB Dataset.csv')

In [21]:
# Display first 5 rows
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [22]:

# Check number of rows and columns
print("Shape:", df.shape)

Shape: (50000, 2)


In [23]:
# Check column names
print("Columns:", df.columns.tolist())

Columns: ['review', 'sentiment']


In [24]:
# Check data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


#**Data cleaning**

In [25]:
df.isnull().sum()

,0
review,0
sentiment,0


**Observation**

There is no missing content


**Check duplicate rows**

In [26]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 418


In [27]:
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

**Observation**

There are totla 418 duplicate values are available so then i remove that values

In [28]:
print(df['sentiment'].value_counts())
print(df['sentiment'].unique())

sentiment
positive    24884
negative    24698
Name: count, dtype: int64
['positive' 'negative']


In [29]:
stop_words = set(stopwords.words('english'))

# Preserve important negation words
stop_words -= {'no', 'nor', 'not'}

lemmatizer = WordNetLemmatizer()


def preprocess_text(text):

    # 1. Handle missing values
    if pd.isna(text):
        return ""

    text = str(text)

    # 2. Decode HTML entities
    text = html.unescape(text)

    # 3. Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # 4. Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # 5. Convert to lowercase
    text = text.lower()

    # 6. Expand common contractions
    text = re.sub(r"\bwon't\b", "will not", text)
    text = re.sub(r"\bcan't\b", "cannot", text)
    text = re.sub(r"\bshan't\b", "shall not", text)
    text = re.sub(r"\bain't\b", "is not", text)
    text = re.sub(r"n't\b", " not", text)
    text = re.sub(r"'re\b", " are", text)
    text = re.sub(r"'ve\b", " have", text)
    text = re.sub(r"'ll\b", " will", text)
    text = re.sub(r"'m\b", " am", text)
    text = re.sub(r"'d\b", " would", text)

    # 7. Remove punctuation, digits and special characters
    text = re.sub(r'[^a-z\s]', ' ', text)

    # 8. Remove extra spaces and tokenize
    tokens = text.split()

    # 9. Remove stopwords while preserving negations
    tokens = [
        word for word in tokens
        if word not in stop_words
    ]

    # 10. Lemmatization
    tokens = [
                lemmatizer.lemmatize(word, pos='v')
                for word in tokens
             ]

    # 11. Return processed text
    return tokens

In [30]:
df['processed_review'] = df['review'].apply(preprocess_text)

In [31]:
df[['review', 'processed_review', 'sentiment']].head()

,review,processed_review,sentiment
0,One of the other reviewers has mentioned that ...,"[one, reviewers, mention, watch, oz, episode, ...",positive
1,A wonderful little production. <br /><br />The...,"[wonderful, little, production, film, techniqu...",positive
2,I thought this was a wonderful way to spend ti...,"[think, wonderful, way, spend, time, hot, summ...",positive
3,Basically there's a family where a little boy ...,"[basically, family, little, boy, jake, think, ...",negative
4,"Petter Mattei's ""Love in the Time of Money"" is...","[petter, mattei, love, time, money, visually, ...",positive


# Word2Vector

In [32]:
# Train Word2Vec Method
word2vec_model = Word2Vec(
                           sentences= df['processed_review'],
                           vector_size= 100,
                           window = 5,
                           min_count= 2,
                           sg = 1,
                           epochs= 10
                        )


In [33]:
# Function to convert a review into an average word vector
def review_to_vector(review, model,vector_size):
  vectors = [model.wv[word] for word in review if word in model.wv]
  if vectors:
    return np.mean(vectors, axis=0)
  else:
    return np.zeros(vector_size)

vector_size = 100
review_vectors = np.array([review_to_vector(review, word2vec_model, vector_size) for review in df['processed_review']])
print(review_vectors)
print(review_vectors.shape)


[[-0.03957998  0.10785922 -0.02703767 ... -0.10613487  0.12503618
  -0.02128335]
 [-0.02721381  0.10623585 -0.05863113 ... -0.12800384 -0.00658612
  -0.01612918]
 [-0.01197268  0.05306442  0.02936392 ... -0.06877647 -0.01441699
  -0.00848291]
 ...
 [ 0.05306103  0.19444208 -0.03481204 ... -0.04424787  0.08663816
   0.0668046 ]
 [-0.06677429  0.19189437 -0.04269221 ... -0.09310533  0.08473229
   0.01175668]
 [-0.06621187  0.07646313 -0.06487761 ...  0.00261467 -0.01552617
   0.11559906]]
(49582, 100)


#**Separate features and target**

In [34]:
X = review_vectors
y = df['sentiment']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (49582, 100)
Target shape: (49582,)


#**Encode sentiment labels**

In [35]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

print(le.classes_)

['negative' 'positive']


**Observation:**

Convert positive and negative labels into numerical values.

#**Split the dataset**

In [36]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
                                                    X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    stratify=y
)
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 39665
Testing samples: 9917


**Observation:**

We'll use 80% for training and 20% for testing, preserving the sentiment proportions with stratify=y.

# Model Building

**Base Model**

In [50]:
lr = LogisticRegression()
lr.fit(X_train, y_train)

LogisticRegression()

**Prediction**

In [51]:
y_pred = lr.predict(X_test)

**Validation**

In [52]:
logistic_f1_score = f1_score(y_test, y_pred)
print('Logistic Regression F1 Score : ', logistic_f1_score)

Logistic Regression F1 Score :  0.8765531062124249


**Support Vector Classifier**

In [53]:
svc= LinearSVC(max_iter=3000)
svc.fit(X_train, y_train)


LinearSVC(max_iter=3000)

**Prediction**

In [54]:
y_pred = svc.predict(X_test)

In [55]:
# Evaluate
svc_f1_score = f1_score(y_test, y_pred)
print('SVC F1 Score : ', svc_f1_score)

SVC F1 Score :  0.8771648813695064


**Decision Tree**

In [56]:
db = DecisionTreeClassifier()
db.fit(X_train, y_train)

DecisionTreeClassifier()

In [57]:
# predict
y_pred = db.predict(X_test)


In [58]:
# Evaluate
decision_tree_f1_score = f1_score(y_test, y_pred)
print('Decision Tree F1 Score :', decision_tree_f1_score)

Decision Tree F1 Score : 0.7405247813411079


**RandomForestClassifier**

In [59]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

RandomForestClassifier()

In [60]:
# predict
y_pred = rf.predict(X_test)


In [61]:
# Evaluate
random_forest_f1_score = f1_score(y_test, y_pred)
print('Random Forest Classifier F1 Score :', random_forest_f1_score)

Random Forest Classifier F1 Score : 0.8493880773786024


**BaggingClassifier**

In [62]:
bc = BaggingClassifier()
bc.fit(X_train, y_train)

BaggingClassifier()

In [63]:
# predict
y_pred = bc.predict(X_test)

In [64]:
# Evaluate
bagging_classifier_f1_score = f1_score(y_test, y_pred)
print('Bagging Calssifier F1 Score : ', bagging_classifier_f1_score)

Bagging Calssifier F1 Score :  0.8064316635745207


**AdaBoostClassifier**

In [65]:
ad = AdaBoostClassifier()
ad.fit(X_train, y_train)

AdaBoostClassifier()

In [66]:
# predict
y_pred = ad.predict(X_test)

In [67]:
# Evaluate
adboost_f1_score = f1_score(y_test, y_pred)
print('Adaboost Classifer F1 Score : ', adboost_f1_score)

Adaboost Classifer F1 Score :  0.8341647154637076


**Gradient Boost Classifier**

In [68]:
gd = GradientBoostingClassifier()
gd.fit(X_train, y_train)

GradientBoostingClassifier()

In [69]:
# Predict
y_pred = gd.predict(X_test)

In [70]:
# Evaluate
gradient_boost_f1_score = f1_score(y_test, y_pred)
print('Gradient Boost Classifier F1 Score :', gradient_boost_f1_score)

Gradient Boost Classifier F1 Score : 0.8500649285785635


# Creating Data Frame for All F1 Scores

In [71]:
df = pd.DataFrame({
                  'Algorithm': [
                              'Logistic Regression',
                              'SVC',
                              'Decision Tree',
                              'Random Forest',
                              'Bagging Classifier',
                              'AdaBoost Classifier',
                              'Gradient Boost Classifier'
                              ],

                  'F1_Score': [
                              logistic_f1_score,
                              svc_f1_score,
                              decision_tree_f1_score,
                              random_forest_f1_score,
                              bagging_classifier_f1_score,
                              adboost_f1_score,
                              gradient_boost_f1_score
                              ]
                  })



In [72]:
df = df.sort_values(by='F1_Score', ascending= False).reset_index(drop=True)
df['F1_Score'] = df['F1_Score']*100
df

,Algorithm,F1_Score
0,SVC,87.716488
1,Logistic Regression,87.655311
2,Gradient Boost Classifier,85.006493
3,Random Forest,84.938808
4,AdaBoost Classifier,83.416472
5,Bagging Classifier,80.643166
6,Decision Tree,74.052478


**Observation:**

By Experimenting all algorithms Support Vector Classifier got highest accuracy Score then i choose svc has the final model.

# Hyperparameter Tuning for SVC

In [85]:
# Defining Svc model
svc = LinearSVC()

In [86]:
# defining parameters
param_grid = {
            'C': [0.001, 0.01, 0.1, 1, 10, 100],
            'loss': ['hinge', 'squared_hinge'],
            'class_weight': [None, 'balanced'],
            'tol': [1e-3, 1e-4]
            }

# GridSearchCv
gs = GridSearchCV(
                  estimator=svc,
                  param_grid=param_grid,
                  cv=3,
                  scoring='f1',
                  n_jobs=-1,
                  verbose=1
                )

gs.fit(X_train, y_train)

Fitting 3 folds for each of 48 candidates, totalling 144 fits


/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


GridSearchCV(cv=3, estimator=LinearSVC(), n_jobs=-1,
             param_grid={'C': [0.001, 0.01, 0.1, 1, 10, 100],
                         'class_weight': [None, 'balanced'],
                         'loss': ['hinge', 'squared_hinge'],
                         'tol': [0.001, 0.0001]},
             scoring='f1', verbose=1)

In [87]:
print('Best Parameters:', gs.best_params_)
print('Best Accuracy:', gs.best_score_)

Best Parameters: {'C': 100, 'class_weight': None, 'loss': 'hinge', 'tol': 0.001}
Best Accuracy: 0.8800376088249848


In [88]:
best_model = gs.best_estimator_

y_pred = best_model.predict(X_test)

In [92]:
final_model_f1_score = f1_score(y_test, y_pred)
print('Final Model F1 Score :',final_model_f1_score )

Final Model F1 Score : 0.8797480810864003


In [93]:
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=le.classes_
))


Classification Report:
              precision    recall  f1-score   support

    negative       0.89      0.86      0.87      4940
    positive       0.86      0.90      0.88      4977

    accuracy                           0.88      9917
   macro avg       0.88      0.88      0.88      9917
weighted avg       0.88      0.88      0.88      9917



# **Save your best model**

In [94]:
import joblib

joblib.dump(best_model, '(Word2Vec&ML)_model.pkl')

word2vec_model.save('imdb_word2vec.model')